# AI Tools for Actuaries
## Exercise of lectures 4 & 5: Feed-Forward Neural Network (FNN)
### Authors: Marco Maggi, Michael Mayer and Mario Wuthrich
### Version Summer School August/September 2026

In [ ]:
import numpy as np
import pandas as pd
import torch

# See all pandas columns
pd.set_option("display.max_columns", None)

## Load data and split into fixed Learn and Test

In [ ]:
df = pd.read_parquet("../../Data/freMTPL2freq.parquet")
df.head()

In [ ]:
learn = ...
test = ...

## Pre-processing

### Define the Scikit-Learn preprocessor

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)


def clip_and_scale(upper):
    pipe = Pipeline(
        steps=[
            ("clip", FunctionTransformer(lambda x: np.clip(x, a_min=0, a_max=upper))),
            ("scale", StandardScaler()),
        ]
    )
    return pipe


density = Pipeline(
    steps=[
        ("log", FunctionTransformer(lambda x: np.log(x).round(2))),
        ...,
    ]
)

area = Pipeline(
    steps=[
        ("encode", OrdinalEncoder()),
        ...,
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "scale",
            clip_and_scale([20, 90, 150, 15]),
            ["VehAge", "DrivAge", "BonusMalus", "VehPower"],
        ),
        ("area", area, ["Area"]),
        ("density", density, ["Density"]),
        ("veh_brand", OneHotEncoder(sparse_output=False), ["VehBrand", "Region"]),
        (
            "veh_gas",
            FunctionTransformer(lambda x: (x == "Diesel").astype(np.float32)),
            ["VehGas"],
        ),
    ],
    verbose_feature_names_out=False,
)

# Just a check: Fit preprocessor to training data and apply to some lines from test
preprocessor.set_output(transform="pandas").fit(learn)
preprocessor.transform(test.head())

### Split in covariates X, responses y, exposures v

In [ ]:
from sklearn.model_selection import train_test_split

convert_to_tensor = lambda x: torch.tensor(x.values, dtype=torch.float32)
train, val = train_test_split(learn, test_size=0.1, random_state=125548)

X_learn = convert_to_tensor(preprocessor.fit_transform(learn))
X_train = convert_to_tensor(preprocessor.transform(train))
X_val = convert_to_tensor(preprocessor.transform(val))
X_test = convert_to_tensor(preprocessor.transform(test))

y_learn, v_learn = convert_to_tensor(learn.ClaimNb), convert_to_tensor(learn.Exposure)
y_train, v_train = convert_to_tensor(train.ClaimNb), convert_to_tensor(train.Exposure)
y_val, v_val = convert_to_tensor(val.ClaimNb), convert_to_tensor(val.Exposure)
y_test, v_test = convert_to_tensor(test.ClaimNb), convert_to_tensor(test.Exposure)

## Define FNN

In [ ]:
from torch import nn
from torch.nn import init


class FNN(nn.Module):
    def __init__(self, seed, n_features, hidden_layers, y0):
        super().__init__()
        torch.manual_seed(seed)
        self.hidden_layers = nn.ModuleList()
        for i in range(len(hidden_layers)):
            if i == 0:
                self.hidden_layers.append(nn.Linear(n_features, hidden_layers[i]))
            else:
                self.hidden_layers.append(
                    nn.Linear(hidden_layers[i - 1], hidden_layers[i])
                )
        self.output_layer = nn.Linear(hidden_layers[-1], 1)
        init.constant_(self.output_layer.weight, 0.0)
        init.constant_(self.output_layer.bias, y0)

    def forward(self, x, v):
        for layer in self.hidden_layers:
            x = torch.tanh(layer(x))
        return torch.exp(self.output_layer(x)).flatten() * v


SEED = 21456783
M_FEAT = X_train.shape[1]  # number of features
HIDDEN = [20, 15, 10]
mu_hom = ...  # homogeneous frequency
# Create model with three hidden layers
model = FNN([...])

## Homogeneous case not considering any covariates

In [ ]:
from sklearn.metrics import mean_poisson_deviance


# Helper functions to evaluate the model via average Poisson deviance
def score(model, X, y, v):
    """Evaluate the model using sklearn's mean_poisson_deviance."""
    pred = model(X, v).detach().numpy()
    return 100 * mean_poisson_deviance([...])


print(f"Poisson Deviance (Learn): {score(model, X_learn, y_learn, v_learn):.3f}")
print(f"Poisson Deviance (Test): {score(model, X_test, y_test, v_test):.3f}")

## Train the model

In [ ]:
def train_model(
    model,
    X_train,
    y_train,
    v_train,
    X_val,
    y_val,
    v_val,
    optimizer,
    checkpoint_path,
    batch_size,
    n_epochs=100,
):
    loss_fn = nn.PoissonNLLLoss(log_input=False, reduction="sum")
    best_val_loss = float("inf")
    history = {"loss": [], "val_loss": []}

    # Create dataset indices for batching
    num_batches = (len(X_train) + batch_size - 1) // batch_size

    for epoch in range(n_epochs):
        # Training phase
        model.train()
        epoch_loss = 0.0
        # indices can be shuffled for each epoch. We don't do it here.
        indices = torch.arange(len(X_train))

        for i in range(num_batches):
            # Get batch indices
            batch_indices = indices[
                i * batch_size : min((i + 1) * batch_size, len(X_train))
            ]

            # Get batch data
            X_batch = X_train[batch_indices]
            v_batch = v_train[batch_indices]
            y_batch = y_train[batch_indices]

            # Forward pass
            pred_batch = model(X_batch, v_batch)
            loss = loss_fn(pred_batch, y_batch)

            # Backward pass and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        # Average loss for the epoch
        epoch_loss /= (
            v_train.sum().item()
        )  # this scaling is different from the Python and R cases
        history["loss"].append(epoch_loss)

        # Validation phase
        model.eval()
        with torch.no_grad():
            pred_val = ...
            val_loss = ...
            history["val_loss"].append(val_loss)

        # Save best model
        if val_loss < best_val_loss and isinstance(checkpoint_path, str):
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)

        # Print progress
        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1}/{n_epochs}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}"
            )
    return history

In [ ]:
optimizer = torch.optim.NAdam(model.parameters())
checkpoint_path = f"./Networks/FNN1_PyTorch_{SEED}.pt"
history = train_model(
    model,
    X_train,
    y_train,
    v_train,
    X_val,
    y_val,
    v_val,
    optimizer,
    checkpoint_path,
    batch_size=5_000,
    n_epochs=100,
)

In [ ]:
# Plot training history (vertical line at best validation loss)
fig = (
    pd.DataFrame({"loss": history["loss"], "val_loss": history["val_loss"]})
    .rename(columns={"loss": "Training", "val_loss": "Validation"})
    .plot(xlabel="Epoch - 1", ylabel="Loss", title="Loss During Training", grid=True)
)
_ = fig.axvline(np.argmin(history["val_loss"]), color="black", linestyle="--")


## Evaluate Results

In [ ]:
# Load best weights and evaluate
model.load_state_dict(torch.load(checkpoint_path))

print(f"Poisson Deviance (Learn): {score(model, X_learn, y_learn, v_learn):.3f}")
print(f"Poisson Deviance (Test): {score(model, X_test, y_test, v_test):.3f}")

model.eval()  # Set model to evaluation mode
with torch.no_grad():
    learn_nn = model(X_learn, v_learn).detach().numpy()

print(f"Balance Property: {mu_hom:.4f} {learn_nn.sum() / learn['Exposure'].sum():.4f}")

## Balance Property Adjustment

In [ ]:
# "Freeze" all parameters of the hidden layers
[el.weight.requires_grad_(requires_grad=False) for el in model.hidden_layers]
[el.bias.requires_grad_(requires_grad=False) for el in model.hidden_layers];

In [ ]:
# we don't do batch training anymore, so we set batch_size to the size of the training
# set
batch_size = len(y_train)
# smaller learning rate since the model is already trained and the balance property is
# only slightly not satisfied
optimizer = torch.optim.NAdam(model.parameters(), lr=0.000001)

history = train_model(
    model,
    X_train,
    y_train,
    v_train,
    X_val,
    y_val,
    v_val,
    optimizer,
    checkpoint_path=None,
    batch_size=batch_size,
    n_epochs=100,
)
_ = pd.DataFrame(history["loss"]).plot(
    xlabel="Epoch - 1",
    ylabel="Loss",
    title="Model Loss During Training",
    grid=True,
    legend=False,
    # ylim=(0, 0.2),
)

In [ ]:
print(f"Poisson Deviance (Learn): {score(model, X_learn, y_learn, v_learn):.3f}")
print(f"Poisson Deviance (Test): {score(model, X_test, y_test, v_test):.3f}")

model.eval()  # Set model to evaluation mode
with torch.no_grad():
    learn_nn_reg = model(X_learn, v_learn).detach().numpy()

print(f"Balance Property: {mu_hom:.4f} {...:.4f}")